##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — Structured Outputs

Force model output to match a Pydantic schema or regex pattern.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/perceptron-mk1/structured-outputs.ipynb)

In [ ]:
%pip install --upgrade perceptron --quiet


## Download assets

In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/qna/studio_scene.webp"

!curl -so studio_scene.webp {IMAGE_URL}

In [ ]:
import os
from pathlib import Path

from IPython.display import Image as IPyImage, display

from perceptron import configure, pydantic_format, regex_format, perceive, image, text

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="perceptron-mk1",
    api_key=api_key,
)

SCENE_PATH = "studio_scene.webp"

## Structured output with Pydantic
Define a Pydantic model and the SDK automatically converts it to a JSON schema for constrained decoding.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class SceneAnalysis(BaseModel):
    """Structured scene analysis output."""

    scene_type: Literal["urban", "nature"]
    main_subjects: list[str] = Field(description="Primary objects in the scene")
    mood: Literal["energetic", "peaceful", "tense"]
    time_of_day: Literal["day", "night", "unknown"]


@perceive(model="perceptron-mk1", response_format=pydantic_format(SceneAnalysis))
def analyze_scene(img_path):
    return image(img_path) + text("Analyze this scene. Output in JSON with scene type, subjects, mood and time of day.")


display(IPyImage(url=IMAGE_URL, width=400))
result = analyze_scene(str(SCENE_PATH))

# Parse directly into the Pydantic model
analysis = SceneAnalysis.model_validate_json(result.text)
print(f"Scene type: {analysis.scene_type}")
print(f"Subjects: {analysis.main_subjects}")
print(f"Mood: {analysis.mood}")
print(f"Time: {analysis.time_of_day}")

## Constrained output with regex

Use a regex pattern for simpler constraints.

In [ ]:
@perceive(model="perceptron-mk1", response_format=regex_format("(energetic|peaceful|tense)"))
def classify_mood(img_path):
    return image(img_path) + text("Analyze this scene and output its mood.")


display(IPyImage(url=IMAGE_URL, width=400))
result = classify_mood(str(SCENE_PATH))

print(f"Mood: {result.text}")